# Linear Regression

Linear regression is an approach we use to model the relationship between a scalar dependent variable *`y`* and one or more explanatory variables (aka independent variables) denoted by *`X`*. In machine learning parlance, these explanatory variables or predictors are called **features** and target or the dependent variable **labels**. Regression modesl are models which predict a continuous outcome.

## Univariate Regression
The simplest possible relationship between two variables is a *`univariate regression`* when there is only one feature to be explained.<br><br>

$$y_i \approx w_0 + w_1x_i$$

If we can measure some $(x_i, y_i)$ pairs, we could calculate our _model parameters_ $w_0$ and $w_1$. Then we could predict $y$ in the future based on $x$, or even try to influence $y$ in the future by controlling $x$. This is achieved by minimizing the total squared error, <br><br>  

$$L = \sum_{i=1}^{n} \left(y_i - w_0 - \sum_{j=1}^{p} x_{ij}w_j \right)^2$$

## Multivariate Regression
If there multiple features that are to be mapped to a target variable, then it is a *`multivariate regression`* problem.<br><br>  

$$ y_i \approx w_0 + w_1x_{1} + w_2x_{2} + w_3x_{3} + \cdots + w_nx_{n} $$

or more generally

$$ y_i \approx  \sum_jw_jX_{ij} $$

where $i$ labels different **observations** and $j$ labels different **features**.

---

_**Note:** Some third-party Python libraries (e.g., yfinance, quantmod) depend on external data sources that may be restricted or inaccessible in certain regions, including China. Users in these regions are advised to consult their local instructor or substitute with locally supported alternatives._

---


**Install Packages**

In [ ]:
# Install packages
!pip install -q quantmod wandb

**Import Libraries**

In [ ]:
# Data Manipulation
import numpy as np
import pandas as pd

# Quantmod
import quantmod.charts
from quantmod.markets import getData
from quantmod.datasets import fetch_historical_data
from quantmod.timeseries import OpCl, HiLo, Gap, dailyReturn, lag, lead
from quantmod.indicators import SMA, BBands


# scikit-learn
import sklearn

# Data Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import Pipeline

# Regressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet

# Metrics
from sklearn.metrics import mean_squared_error

# Import Libraries
from google.colab import userdata

## Section 1: Experiment Tracking

Experiment tracking is a foundational practice in data science, machine learning, and quantitative research. As experiments grow in complexity - spanning multiple datasets, models, hyperparameters, and evaluation metrics - it becomes increasingly difficult to reproduce results or understand what actually worked. This is where experiment trackers come in.

> _An experiment tracker is a tool or framework that systematically records every aspect of an experiment: code versions, input data, parameter configurations, outputs, metrics, and environment details. Instead of relying on ad-hoc notes or scattered logs, it enables a structured, searchable history of our work, improving reproducibility while accelerating iteration and collaboration._

Popular platforms such as MLflow and Weights & Biases provide both local and cloud-based solutions, supporting everything from lightweight logging to full-scale experiment management.

In our labs, we use Weights & Biases (**W&B**), which integrates seamlessly with Python workflows, enabling real-time logging, run comparison, and artifact management. It brings discipline and visibility to experimentation, making iteration faster and results more reliable.

Create an account on [Weights & Biases](https://wandb.ai/), set up a project, and generate your API key to get started. Save it in Colab Secrets to enable seamless authentication.

In [ ]:
# Experiment Tracker
import wandb

# Login to WandB & get API Key
wandb.login(key=userdata.get('WANDB_API_KEY'))

## Section 2: The workflow

We'll employ the following linear regressor from `scikit-learn` for price prediction.

* Linear Regression
* Lasso Regression
* Ridge Regression
* Elastic Net Regression

| Steps        | Workflow                  | Remarks                                                         |
|:-------------|:--------------------------|:----------------------------------------------------------------|
|Step 1        | Ideation                  | Define objective, success metrics     |
|Step 2        | Data Collection           | Gather and integrate data
|Step 3        | Exploratory Data Analysis (Initial) | Broad exploration: stats, distributions, correlations, missing data |
|Step 4        | Data Cleaning           | Handle missing values, outliers, duplicates.            |
|Step 5        | Feature Engineering & Transformation            | Feature creation, scaling, encoding, selection                         |
|        | Subset Validation EDA            | Re-examine chosen features: check distributions, multicollinearity, relationships                      |               
|Step 6        | Modeling                  | Select algorithm(s), train models, tune hyperparameters                           |
|Step 7        | Evaluation                   | Validate using metrics and backtesting       |

### (1) Load Data

In [ ]:
# Load sample data from quandmod
df = (
    fetch_historical_data("NIFTY")
    .assign(date=lambda df_: pd.to_datetime(df_["date"]))
    .set_index('date')
    .copy()
)

df

In [ ]:
df.index

### (2) EDA of Original dataset

In [ ]:
# Descriptive Statistics
df.describe().T

### (3) Cleaning & Imputation

Data is already cleaned. No further processing or imputation required.

In [ ]:
# Check for missing values
df.isnull().sum()

### (4) Feature Engineering
Features are also known as an independent variable which are used to determine the value of the target variable. We will now create features and target (label) from the raw dataset.

In [ ]:
# Define Create Features
def create_features(frame):
    df = frame.copy()
    multiplier = 2

    # Features - X
    df['OC'] = OpCl(df)
    df['HC'] = HiLo(df)
    df['GAP'] = Gap(df)
    df['RET'] = dailyReturn(df.close)

    for i in [7, 14, 28]:
        df['PCHG' + str(i)] = df['close'].pct_change(i)
        df['VCHG' + str(i)] = df['volume'].pct_change(i)
        df['RET' + str(i)] = df['RET'].rolling(i).sum()
        df['MA' + str(i)] = SMA(df['close'], i)
        # df['VMA' + str(i)] = SMA(df['volume'], i)
        df['OC'+ str(i)] = df['OC'].rolling(i).mean()
        df['HC'+ str(i)] = df['HC'].rolling(i).mean()
        df['GAP'+ str(i)] = df['GAP'].rolling(i).mean()
        df['STD'+ str(i)] = df['RET'].rolling(i).std()
        df['LB'+str(i)] = BBands(df['close'], i, multiplier)[0]
        df['UB'+str(i)] = BBands(df['close'], i, multiplier)[2]

    # Define the label - y
    df['Label'] = lead(df['close'])

    # Drop unnecessary columns - important
    df.drop(['open', 'high', 'low', 'close', 'volume'], axis=1, inplace=True)
    df.dropna(inplace=True)

    return df

#### (a) Feature Specification

In [ ]:
# Features
df1 = create_features(df)
display(df1.shape)

# Verify the output
df1.head(2)

#### (b) Target or Label Definition
Label or the target variable is also known as the dependent variable. Here, the target variable is the closing price of the index the next trading day.

In [ ]:
# Label
y = np.array(df1['Label'])
y

#### (c) Feature Selection
Reduce the number of input variables that are believed to be most useful to a model. We use correlation measures which is one the filter methods to address multicollinearity among features.

In [ ]:
# Drop label from dataframe
df1.drop('Label', axis=1, inplace=True)

# Plot correlation matrix
df1.corr().iplot(
    kind="heatmap",
    title="Features Set Correlations",
    width=1000,
    height=800
    )

In [ ]:
# Remove highly correlated features - greater than 0.90 in this case
threshold = 0.9
corr_matrix = df1.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find columns to drop
to_drop = [column for column in upper.columns if any(upper[column] > threshold)]

# Number of features dropped
print(f"\n No. of features dropped: {len(to_drop)} \n List of features dropped: {to_drop}")

**Selected Features**

In [ ]:
# Drop the highly correlated features
X = df1.drop(to_drop, axis=1)

# record feature names
feature_names = X.columns

In [ ]:
# Display the new features set
X.head()

### (5) EDA of Subset Selection

> The choice of scaling techniques to be used should come from the Exploratory Data Analysis of features set.

In [ ]:
# decide which scaling to use
X.describe()

In [ ]:
X.iplot(
    kind="box",
    showlegend=True,
    title="Box Plot Analysis",
    width=1500,
    height=800
    )

### (6) Linear Regression
Linear regression is an approach we use to model the relationship between a scalar dependent variable *`y`* and one or more explanatory variables (aka independent variables) denoted by *`X`*.

**Train - Test Data**

In [ ]:
# Splitting the datasets into training and testing data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Output the train and test data size
print(f"Train and Test Size {len(X_train)}, {len(X_test)}")

In [ ]:
# Use pipeline to define the linear regression model
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

lr

In [ ]:
# Fit/Train model
lr.fit(X_train, y_train)

# Predict labels
y_pred = lr.predict(X_test)

# Metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f'MSE: {mse:0.4}')
print(f'RMSE: {rmse:0.4}')
print(f'R^2 Train: {lr.score(X_train, y_train):0.4}')
print(f'R^2 Test: {lr.score(X_test, y_test):0.4}')

In [ ]:
# Get the coefficients or weights of the model
lr_coef = lr['regressor'].coef_
lr_coef

In [ ]:
# Get the intercept of the regression model
lr['regressor'].intercept_

**Regularized Regression**

If the coeficients are too large in linear regression, it can lead to overfitting the model. We do regularization to over come such issues that penalises large coefficients. Lasso and Ridge regression are penalty regression that prevent over-fitting from the simple linear regression.

#### (a) LASSO
The Least Absolute Shrinkage and Selection Operator (LASSO) is a variation of linear regression. In Lasso, the loss function is minimized by limiting the sum of absolute values of the model coefficients where the L1 penalty term is added to the Residual Sum of Squares (RSS). The L1 penalty term not only shrinks the coefficents, but shrinks some of them to zero and that is very useful for feature selection. <br><br>

$$L = \sum_{i=1}^{n} \left(y_i - w_0 - \sum_{j=1}^{p} x_{ij}w_j \right)^2  + \lambda \sum_{j=1}^{p} {\mid w_j\mid}$$

where,  $\lambda$ is the regularization penalty.

In [ ]:
# Use pipeline to define the Lasso model
lasso = Pipeline([
    ('scaler', MinMaxScaler()),
    ('regressor', Lasso(alpha=0.1))
])

# Fit lasso model
lasso.fit(X_train, y_train)

# Prediction
lasso_pred = lasso.predict(X_test)

print(f'R^2 Train: {lasso.score(X_train, y_train):0.4}')
print(f'R^2 Test: {lasso.score(X_test, y_test):0.4}')

In [ ]:
# Get the coefficients or weights of the model
lasso_coef = lasso['regressor'].coef_
lasso_coef

In [ ]:
# Log Lasso coefficients vs regularization to wandb
run = wandb.init(
    project="linear-regression-demo-2",
    name="lasso-coef-path",
    job_type="coef-path",
    config={
        "model": "Lasso",
        "scaler": "StandardScaler",
        "alpha_min": 1e-2,
        "alpha_max": 1e2,
        "n_alphas": 200
        }
    )

coef = []
alpha_range = np.logspace(2, -2, 200)

for alpha in alpha_range:
    las = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', Lasso(alpha=alpha, max_iter=10000))
    ])
    las.fit(X_train, y_train)
    y_pred = las.predict(X_test)

    w = las.named_steps['regressor'].coef_
    coef.append(w)

    log = {"alpha": alpha, "log_alpha": np.log10(alpha),
           "train_r2": las.score(X_train, y_train),
           "test_r2": las.score(X_test, y_test),
           "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
           "n_nonzero": int(np.sum(w != 0))}

    for name, val in zip(feature_names, w):
        log[f"coef/{name}"] = float(val)

    wandb.log(log)

coef_df = pd.DataFrame(coef, columns=feature_names, index=alpha_range)
coef_df.index.name = 'alpha'

# Log the full coefficient path as a table
wandb.log({"lasso_coef_path": wandb.Table(dataframe=coef_df.reset_index())})
run.finish()

#### (b) Ridge
In ridge regression, the cost function is altered by adding a L2 penality equivalent to square of the magnitude of the coefficients. The Ridge regression shrinks the coefficients and helps to reduce the multi-collinearity. As seen from the above graph, when $\lambda \xrightarrow {}0$, the cost function becomes similar to the linear regression cost function RSS.

$$L = \sum_{i=1}^{n} \left(y_i - w_0 - \sum_{j=1}^{p} x_{ij}w_j \right)^2  + \lambda \sum_{j=1}^{p} w_j^2$$

In [ ]:
# Use pipeline to define the ridge model
rid = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Ridge(alpha=1))
])

# fit ridge model
rid.fit(X, y)

# predict
ridge_pred = rid.predict(X_test)

print(f'R^2 Train: {rid.score(X_train, y_train):0.4}')
print(f'R^2 Test: {rid.score(X_test, y_test):0.4}')

In [ ]:
# Get the coefficients or weights of the model
ridge_coef = rid['regressor'].coef_
ridge_coef

**Coefficient Analysis**

In [ ]:
coef = []
alpha_range = np.logspace(6, -2, 200)

for alpha in alpha_range:
    # Define pipeline
    rid = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', Ridge(alpha=alpha, max_iter=10000))
    ])

    # Fit pipeline on training data
    rid.fit(X_train, y_train)

    # Predict on test set if needed (optional)
    y_pred = rid.predict(X_test)

    # Store coefficients
    coef.append(rid.named_steps['regressor'].coef_)

# Create DataFrame: rows = alphas, columns = features
coef_df = pd.DataFrame(coef, columns=feature_names, index=alpha_range)
coef_df.index.name = 'alpha'


In [ ]:
# plot coefficients
# coef_df.iplot(kind='overlay', showlegend=True)
fig = coef_df.drop('MA7', axis=1).iplot(
    kind='overlay',
    title='Coefficients as a function of the regularization',
    xaxis_title='λ (log scale)',
    yaxis_title='w',
    showlegend=True
    )

# plot figure
fig.show()

In [ ]:
# Log Ridge coefficients vs regularization to wandb
run = wandb.init(
    project="linear-regression-demo-2",
    name="ridge-coef-path",
    job_type="coef-path",
    config={
        "model": "Ridge",
        "scaler": "StandardScaler",
        "alpha_min": 1e-2,
        "alpha_max": 1e6,
        "n_alphas": 200
        }
    )

coef = []
alpha_range = np.logspace(6, -2, 200)

for alpha in alpha_range:
    rid = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', Ridge(alpha=alpha, max_iter=10000))
    ])
    rid.fit(X_train, y_train)
    y_pred = rid.predict(X_test)

    w = rid.named_steps['regressor'].coef_
    coef.append(w)

    log = {"alpha": alpha, "log_alpha": np.log10(alpha),
           "train_r2": rid.score(X_train, y_train),
           "test_r2": rid.score(X_test, y_test),
           "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred))}

    for name, val in zip(feature_names, w):
        log[f"coef/{name}"] = float(val)

    wandb.log(log)

coef_df = pd.DataFrame(coef, columns=feature_names, index=alpha_range)
coef_df.index.name = 'alpha'

wandb.log({"ridge_coef_path": wandb.Table(dataframe=coef_df.reset_index())})
wandb.log({"Coeff": fig})

run.finish()

#### (c) ElasticNet Regression
ElasticNet combines the properties of both Lasso and Ridge regression. It penalizes the model using both the L1 and L2 norm. <br><br>

$$L = \sum_{i=1}^{n} \left(y_i - w_0 - \sum_{j=1}^{p} x_{ij}w_j \right)^2  + \lambda \left(\frac{1-\alpha}{2} \sum_{j=1}^{p} w_j^2 + \alpha \sum_{j=1}^{p} {\mid w_j\mid} \right)$$

where $0\leq \alpha \leq1$. Elastic net is same as lasso when $\alpha=1$ and ridge when $\alpha=0$. For other values of $\alpha$, the penalty term interpolate between L1 and L2 norm of the coefficient.

In [ ]:
# Use pipeline to define the elastic net model
elastic = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', ElasticNet(alpha=0.1, l1_ratio=0.3))
])

# fit elasticnet model
elastic.fit(X_train, y_train)

# predict
elasticnet_pred = elastic.predict(X_test)

print(f'R^2 Train: {elastic.score(X_train, y_train):0.4}')
print(f'R^2 Test: {elastic.score(X_test, y_test):0.4}')

In [ ]:
# Get the coefficients or weights of the model
elasticnet_coef = elastic['regressor'].coef_
elasticnet_coef

In [ ]:
# Log ElasticNet coefficients vs regularization to wandb
run = wandb.init(
    project="linear-regression-demo-2",
    name="elasticnet-coef-path",
    job_type="coef-path",
    config={
        "model": "ElasticNet",
        "scaler": "StandardScaler",
        "l1_ratio": 0.5,
        "alpha_min": 1e-2,
        "alpha_max": 1e2,
        "n_alphas": 200
        }
    )

coef = []
alpha_range = np.logspace(2, -2, 200)

for alpha in alpha_range:
    en = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', ElasticNet(alpha=alpha, max_iter=10000))
    ])
    en.fit(X_train, y_train)
    y_pred = en.predict(X_test)

    w = en.named_steps['regressor'].coef_
    coef.append(w)

    log = {"alpha": alpha, "log_alpha": np.log10(alpha),
           "train_r2": en.score(X_train, y_train),
           "test_r2": en.score(X_test, y_test),
           "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
           "n_nonzero": int(np.sum(w != 0))}

    for name, val in zip(feature_names, w):
        log[f"coef/{name}"] = float(val)

    wandb.log(log)

coef_df = pd.DataFrame(coef, columns=feature_names, index=alpha_range)
coef_df.index.name = 'alpha'

wandb.log({"elasticnet_coef_path": wandb.Table(dataframe=coef_df.reset_index())})
run.finish()

### (7) Evaluation: Model Comparision

In [ ]:
# compare model coefficients
# weights[weights['Lasso']!=0].index

weights = pd.DataFrame({
    'LR': lr_coef,
    'Lasso': lasso_coef,
    'Ridge': ridge_coef,
    'ElasticNet': elasticnet_coef
}, index = feature_names)

weights

## Section 3: Also Read

📌 [Quantmod](https://kannansingaravelu.com/quantmod/)

📌 [Scikit-learn](https://scikit-learn.org/stable/index.html)

📌 [Weights & Biases](https://docs.wandb.ai/)  

---
[Kannan Singaravelu](https://www.linkedin.com/in/kannansi)